# 05b · Probing: how much do the features know about the background?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/05b_probing.ipynb)

**Runtime → T4 GPU → Run all.** ~40–60 мин. Ничего не обучает, кроме маленьких линейных классификаторов.

Берём признаки из предпоследнего слоя каждой модели на test-картинках с заменённым фоном и учим логистическую
регрессию угадывать **категорию сцены фона** (87 категорий Places365, случайное угадывание ≈ 1.1%), 5-fold CV.
Фон назначен случайно и с породой не связан, поэтому для распознавания пород эта информация бесполезна.
Чем лучше её можно достать из признаков, тем больше модель «держит в голове» фон.

Для сравнения — те же сети с ImageNet-весами без дообучения и случайно инициализированная ResNet-18.

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from bgbias import analysis as A
from bgbias.models import build_model, load_checkpoint, strip_head
from bgbias.project import make_dataset
from bgbias.train import make_loader

device = torch.device("cuda")
SPLIT = "test"
scenes = data.manifests[SPLIT].set_index("image_id").loc[data.splits[SPLIT].image_id, "bg_category"].to_numpy()
print(len(set(scenes)), "scene categories, chance =", f"{1 / len(set(scenes)):.1%}")
loader = make_loader(make_dataset(data, SPLIT, condition="swapped"), 128, False, 0, 2)

@torch.no_grad()
def features(model):
    model.eval()
    out = []
    for x, _ in loader:
        with torch.autocast("cuda"):
            out.append(model(x.to(device, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy()

def probe(feats):
    clf = make_pipeline(StandardScaler(), PCA(128, random_state=0), LogisticRegression(max_iter=2000, C=0.5))
    return cross_val_score(clf, feats, scenes, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()

In [ ]:
CACHE = DRIVE_ROOT / "results" / "analysis" / "probing_runs.csv"   # finished runs are skipped on a rerun
CACHE.parent.mkdir(parents=True, exist_ok=True)
done = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame(columns=["run", "model", "scene_acc"])

def record(run, model_name, make_model):
    global done
    if run in set(done.run):
        return
    model = make_model().to(device)
    acc = probe(features(model))
    done = pd.concat([done, pd.DataFrame([{"run": run, "model": model_name, "scene_acc": acc}])], ignore_index=True)
    done.to_csv(CACHE, index=False)
    print(f"{run:24s} scene accuracy {acc:.1%}")
    del model
    torch.cuda.empty_cache()

# references: ImageNet features without fine-tuning, and an untrained network
for arch in ("resnet18", "resnet50", "convnext_tiny", "swin_t"):
    record(f"imagenet_{arch}", f"imagenet_{arch}", lambda: strip_head(build_model(arch, data.num_classes), arch))
record("random_resnet18", "random_resnet18",
       lambda: strip_head(build_model("resnet18", data.num_classes, pretrained=False), "resnet18"))

for run in sorted(p.parent for p in RUNS.glob("*_seed*/best.pt")):
    def make(run=run):
        model, cfg = load_checkpoint(run / "best.pt", data.num_classes, device)
        return strip_head(model, cfg.get("arch", "resnet18"))
    record(run.name, A.model_name(run.name), make)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from bgbias.plotting import COLORS, INK2, saver, use_style

use_style()
FIG_DIR = Path(REPO_DIR) / "report" / "figures"
TAB = Path(REPO_DIR) / "results" / "analysis"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
save = saver(FIG_DIR)

stats = done.groupby("model").scene_acc.agg(["mean", "std", "count"])
stats["std"] = stats["std"].fillna(0)   # references have a single run
stats.to_csv(TAB / "probing.csv")
REF = {"ResNet-18": "imagenet_resnet18", "ResNet-18 (scratch)": "random_resnet18", "ResNet-50": "imagenet_resnet50",
       "ConvNeXt-T": "imagenet_convnext_tiny", "Swin-T": "imagenet_swin_t"}
fams = [f for f, (ma, mb) in A.FAMILIES.items() if ma in stats.index and mb in stats.index]
meths = [(label, m) for label, m in A.METHODS.items() if m in stats.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 0.55 * max(len(fams), len(meths)) + 1.8))
ax = axes[0]
y = np.arange(len(fams))
for variant, key, dy in ((0, "resnet_A", -0.12), (1, "resnet_B", 0.12)):
    m = [A.FAMILIES[f][variant] for f in fams]
    ax.errorbar(stats.loc[m, "mean"], y + dy, xerr=stats.loc[m, "std"].fillna(0), fmt="o", ms=7,
                color=COLORS[key], capsize=3, label="A: original images" if variant == 0 else "B: background swap")
ref = [stats.loc[REF[f], "mean"] if REF[f] in stats.index else np.nan for f in fams]
ax.plot(ref, y, "|", ms=14, mew=2, color=INK2, label="before fine-tuning (ImageNet / random)")
ax.set_yticks(y, fams)
ax.invert_yaxis()
ax.set_title("Architectures")
ax.legend(loc="lower right", fontsize=8)
ax = axes[1]
y = np.arange(len(meths))
ax.errorbar([stats.loc[m, "mean"] for _, m in meths], y, xerr=[stats.loc[m, "std"] for _, m in meths],
            fmt="o", ms=7, color=INK2, capsize=3)
ax.set_yticks(y, [label for label, _ in meths])
ax.invert_yaxis()
ax.set_title("Methods (ResNet-18)")
for ax in axes:
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", visible=True, color="#ebeae6")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xlabel("Background scene decoded from features (linear probe, 87 classes)")
fig.tight_layout()
save(fig, "fig7_probing")
stats.round(3)

In [ ]:
import shutil
from bgbias.github_sync import sync

shutil.copytree(FIG_DIR, DRIVE_ROOT / "figures", dirs_exist_ok=True)
sync(REPO_DIR, "Probing: background scene decodability from features")